In [2]:
import sys

sys.path.append(
    "/home/jupyter-pushpashrip.23cse/.local/lib/python3.12/site-packages"
)

sys.path.append(
    "/opt/tljh/user/lib/python3.12/site-packages"
)

print("Package paths added")

Package paths added


In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

print("TensorFlow:", tf.__version__)
print("Pandas:", pd.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

I0000 00:00:1788423959.820009 3284697 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1788423959.880986 3284697 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 AMX_FP16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1788423961.428842 3284697 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow: 2.21.0
Pandas: 3.0.3
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
DATASET_PATH = "/home/jupyter-pushpashrip.23cse/datasets/Glaucoma/Research_Project/projectwork2-phase1/Dataset/processed_dataset.csv"

df = pd.read_csv(DATASET_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nLabel distribution:")
print(df["label_name"].value_counts())

Dataset shape: (13741, 9)

Columns:
['image_name', 'processed_image_path', 'label', 'label_name', 'dataset', 'brightness', 'contrast', 'sharpness', 'preprocessing']

Label distribution:
label_name
Normal      8581
Glaucoma    5160
Name: count, dtype: int64


In [5]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))

print("\nTraining labels:")
print(train_df["label_name"].value_counts())

print("\nValidation labels:")
print(val_df["label_name"].value_counts())

print("\nTesting labels:")
print(test_df["label_name"].value_counts())

Training: 9618
Validation: 2061
Testing: 2062

Training labels:
label_name
Normal      6006
Glaucoma    3612
Name: count, dtype: int64

Validation labels:
label_name
Normal      1287
Glaucoma     774
Name: count, dtype: int64

Testing labels:
label_name
Normal      1288
Glaucoma     774
Name: count, dtype: int64


In [6]:
import os

print("Missing training images:", 
      sum(not os.path.exists(p) for p in train_df["processed_image_path"]))

print("Missing validation images:", 
      sum(not os.path.exists(p) for p in val_df["processed_image_path"]))

print("Missing testing images:", 
      sum(not os.path.exists(p) for p in test_df["processed_image_path"]))

Missing training images: 9618
Missing validation images: 2061
Missing testing images: 2062


In [7]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nFirst image path from CSV:")
print(df["processed_image_path"].iloc[0])

print("\nDoes first image exist?")
print(os.path.exists(df["processed_image_path"].iloc[0]))

Current working directory:
/srv/data/datasets/Glaucoma/Research_Project/projectwork2-phase1/Model_Training

First image path from CSV:
/content/drive/MyDrive/Research_Project/Processed_Dataset/ORIGA/072.jpg

Does first image exist?
False


In [8]:
import os

image_name = df["image_name"].iloc[0]

print("Image name:", image_name)

# Search under the project directory
project_path = "/home/jupyter-pushpashrip.23cse/datasets/Glaucoma/Research_Project/projectwork2-phase1"

matches = []

for root, dirs, files in os.walk(project_path):
    if image_name in files:
        matches.append(os.path.join(root, image_name))

print("\nMatches found:")
for path in matches[:10]:
    print(path)

print("\nNumber of matches:", len(matches))

Image name: 072.jpg

Matches found:
/home/jupyter-pushpashrip.23cse/datasets/Glaucoma/Research_Project/projectwork2-phase1/Processed_Dataset/ORIGA/072.jpg
/home/jupyter-pushpashrip.23cse/datasets/Glaucoma/Research_Project/projectwork2-phase1/Dataset/ORIGA/dataset (divided)/dataset (divided)/Train/yes/072.jpg
/home/jupyter-pushpashrip.23cse/datasets/Glaucoma/Research_Project/projectwork2-phase1/Dataset/ORIGA/dataset/dataset/yes/072.jpg

Number of matches: 3


In [9]:
HPC_PROCESSED_ROOT = (
    "/home/jupyter-pushpashrip.23cse/"
    "datasets/Glaucoma/Research_Project/"
    "projectwork2-phase1/Processed_Dataset"
)

df["hpc_image_path"] = df["processed_image_path"].str.replace(
    "/content/drive/MyDrive/Research_Project/Processed_Dataset",
    HPC_PROCESSED_ROOT,
    regex=False
)

print("Original path:")
print(df["processed_image_path"].iloc[0])

print("\nHPC path:")
print(df["hpc_image_path"].iloc[0])

print("\nDoes HPC image exist?")
print(os.path.exists(df["hpc_image_path"].iloc[0]))

Original path:
/content/drive/MyDrive/Research_Project/Processed_Dataset/ORIGA/072.jpg

HPC path:
/home/jupyter-pushpashrip.23cse/datasets/Glaucoma/Research_Project/projectwork2-phase1/Processed_Dataset/ORIGA/072.jpg

Does HPC image exist?
True


In [10]:
missing = df[~df["hpc_image_path"].apply(os.path.exists)]

print("Total images:", len(df))
print("Existing images:", len(df) - len(missing))
print("Missing images:", len(missing))

if len(missing) > 0:
    print("\nFirst missing paths:")
    print(missing[["image_name", "hpc_image_path"]].head(10))

Total images: 13741
Existing images: 13741
Missing images: 0


In [11]:
IMG_SIZE = 300
BATCH_SIZE = 8

print("Image size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)

Image size: 300
Batch size: 8


In [12]:
def load_image(path, label):
    image = tf.io.read_file(path)

    image = tf.image.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE]
    )

    image = tf.cast(image, tf.float32)

    return image, label

print("Image loading function created successfully.")

Image loading function created successfully.


In [13]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip(mode="horizontal"),
    tf.keras.layers.RandomRotation(factor=0.08),
    tf.keras.layers.RandomZoom(
        height_factor=0.10,
        width_factor=0.10
    ),
    tf.keras.layers.RandomTranslation(
        height_factor=0.05,
        width_factor=0.05
    ),
    tf.keras.layers.RandomContrast(factor=0.10)
], name="data_augmentation")

print("Data augmentation created successfully.")

Data augmentation created successfully.


I0000 00:00:1788424020.071341 3284697 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46069 MB memory:  -> device: 0, name: NVIDIA H200 NVL, pci bus id: 0000:0b:00.0, compute capability: 9.0a


In [14]:
def create_dataset(dataframe, training=False):
    paths = dataframe["hpc_image_path"].values
    labels = dataframe["label"].values.astype(np.float32)

    dataset = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    dataset = dataset.map(
        load_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            seed=42
        )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset


train_dataset = create_dataset(
    train_df,
    training=True
)

val_dataset = create_dataset(
    val_df,
    training=False
)

test_dataset = create_dataset(
    test_df,
    training=False
)

print("Datasets created successfully.")

KeyError: 'hpc_image_path'

In [15]:
train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["hpc_image_path"] = df.loc[
    train_df.index, "hpc_image_path"
]

val_df["hpc_image_path"] = df.loc[
    val_df.index, "hpc_image_path"
]

test_df["hpc_image_path"] = df.loc[
    test_df.index, "hpc_image_path"
]

print("Training paths:", len(train_df))
print("Validation paths:", len(val_df))
print("Testing paths:", len(test_df))

print("\nExample HPC path:")
print(train_df["hpc_image_path"].iloc[0])

print("\nPath exists:")
print(os.path.exists(train_df["hpc_image_path"].iloc[0]))

Training paths: 9618
Validation paths: 2061
Testing paths: 2062

Example HPC path:
/home/jupyter-pushpashrip.23cse/datasets/Glaucoma/Research_Project/projectwork2-phase1/Processed_Dataset/SMDG/OIA-ODIR-TRAIN-2851.png

Path exists:
True


In [16]:
def create_dataset(dataframe, training=False):
    paths = dataframe["hpc_image_path"].values
    labels = dataframe["label"].values.astype(np.float32)

    dataset = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    dataset = dataset.map(
        load_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            seed=42
        )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset


train_dataset = create_dataset(train_df, training=True)
val_dataset = create_dataset(val_df, training=False)
test_dataset = create_dataset(test_df, training=False)

print("Datasets created successfully.")

E0000 00:00:1788424042.898432 3393494 ptx_compiler_helpers.cc:154] *** WARNING *** Invoking ptxas with version 12.0.140, which corresponds to a CUDA version <=12.6.2. CUDA versions 12.x.y up to and including 12.6.2 miscompile certain edge cases around clamping.
Please upgrade to CUDA 12.6.3 or newer.


Datasets created successfully.


In [17]:
def augment_batch(images, labels):
    images = data_augmentation(
        images,
        training=True
    )
    return images, labels


train_dataset = train_dataset.map(
    augment_batch,
    num_parallel_calls=tf.data.AUTOTUNE
)

print("Training augmentation applied.")

Training augmentation applied.


In [18]:
images, labels = next(iter(train_dataset))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Image dtype:", images.dtype)
print("Label dtype:", labels.dtype)
print("Image value range:", tf.reduce_min(images).numpy(), "to", tf.reduce_max(images).numpy())


Image batch shape: (8, 300, 300, 3)
Label batch shape: (8,)
Image dtype: <dtype: 'float32'>
Label dtype: <dtype: 'float32'>
Image value range: 0.0 to 255.0


In [19]:
MODEL_PATH = (
    "/home/jupyter-pushpashrip.23cse/"
    "datasets/Glaucoma/Research_Project/"
    "projectwork2-phase1/Model_Training/"
    "efficientnetb3_glaucoma_final.keras"
)

model = tf.keras.models.load_model(MODEL_PATH)

print("Model loaded successfully.")
print("Total parameters:", model.count_params())

Model loaded successfully.
Total parameters: 10785072


In [20]:
print("Outer model layers:", len(model.layers))

for i, layer in enumerate(model.layers):
    print(i, layer.name, type(layer).__name__)

Outer model layers: 5
0 input_layer_2 InputLayer
1 efficientnetb3 Functional
2 global_average_pooling2d GlobalAveragePooling2D
3 dropout Dropout
4 dense Dense


In [21]:
base_model = model.layers[1]

print("Base model name:", base_model.name)
print("Total EfficientNet layers:", len(base_model.layers))

trainable_count = sum(layer.trainable for layer in base_model.layers)
frozen_count = len(base_model.layers) - trainable_count

print("Currently trainable EfficientNet layers:", trainable_count)
print("Currently frozen EfficientNet layers:", frozen_count)

print("\nLast 10 EfficientNet layers:")

for i, layer in enumerate(base_model.layers[-10:], start=len(base_model.layers)-10):
    print(i, layer.name, "Trainable:", layer.trainable)

Base model name: efficientnetb3
Total EfficientNet layers: 385
Currently trainable EfficientNet layers: 31
Currently frozen EfficientNet layers: 354

Last 10 EfficientNet layers:
375 block7b_se_reduce Trainable: True
376 block7b_se_expand Trainable: True
377 block7b_se_excite Trainable: True
378 block7b_project_conv Trainable: True
379 block7b_project_bn Trainable: True
380 block7b_drop Trainable: True
381 block7b_add Trainable: True
382 top_conv Trainable: True
383 top_bn Trainable: True
384 top_activation Trainable: True


In [22]:
# Freeze all EfficientNet layers first
base_model.trainable = True

for layer in base_model.layers[:-50]:
    layer.trainable = False

# Keep the last 50 layers trainable
for layer in base_model.layers[-50:]:
    layer.trainable = True

trainable_count = sum(layer.trainable for layer in base_model.layers)
frozen_count = len(base_model.layers) - trainable_count

print("Total EfficientNet layers:", len(base_model.layers))
print("Trainable EfficientNet layers:", trainable_count)
print("Frozen EfficientNet layers:", frozen_count)

Total EfficientNet layers: 385
Trainable EfficientNet layers: 50
Frozen EfficientNet layers: 335


In [23]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc")
    ]
)

print("Model compiled successfully.")

Model compiled successfully.


In [24]:
EXPERIMENT_2_MODEL_PATH = (
    "/home/jupyter-pushpashrip.23cse/"
    "datasets/Glaucoma/Research_Project/"
    "projectwork2-phase1/Model_Training/"
    "efficientnetb3_glaucoma_experiment2.keras"
)

callbacks_exp2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_auc",
        patience=5,
        mode="max",
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        EXPERIMENT_2_MODEL_PATH,
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    )
]

print("Experiment 2 callbacks created successfully.")
print("Best model will be saved to:")
print(EXPERIMENT_2_MODEL_PATH)

Experiment 2 callbacks created successfully.
Best model will be saved to:
/home/jupyter-pushpashrip.23cse/datasets/Glaucoma/Research_Project/projectwork2-phase1/Model_Training/efficientnetb3_glaucoma_experiment2.keras


In [25]:
history_exp2 = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=50,
    callbacks=callbacks_exp2
)

Epoch 1/50


/opt/venvs/deeplearning/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1788424150.974158 3384343 service.cc:153] XLA service 0x709b78099070 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1788424150.974199 3384343 service.cc:161]   StreamExecutor [0]: NVIDIA H200 NVL, Compute Capability 9.0a (Driver: 13.0.0; Runtime: 12.0.0; Toolkit: 12.5.0; DNN: 9.20.0)
I0000 00:00:1788424151.698595 3384343 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1788424155.173040 3384343 cuda_dnn.cc:461] Loaded cuDNN version 92000
E0000 00:00:1788424177.606157 3384343 cuda_timer.cc:87] Delay kernel timed ou

1202/1203 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.8045 - auc: 0.8648 - loss: 0.4339

E0000 00:00:1788424398.718921 3384345 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1788424404.595928 3384345 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1788424420.124550 3384345 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1788424427.725799 3384345 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1788424454.133948 3384345 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000

1203/1203 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - accuracy: 0.8044 - auc: 0.8647 - loss: 0.4339

E0000 00:00:1788424526.280273 3384344 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1788424532.606884 3384344 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1788424551.740834 3384344 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1788424551.961403 3384344 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1788424552.200778 3384344 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000


Epoch 1: val_auc improved from None to 0.91484, saving model to /home/jupyter-pushpashrip.23cse/datasets/Glaucoma/Research_Project/projectwork2-phase1/Model_Training/efficientnetb3_glaucoma_experiment2.keras

Epoch 1: finished saving model to /home/jupyter-pushpashrip.23cse/datasets/Glaucoma/Research_Project/projectwork2-phase1/Model_Training/efficientnetb3_glaucoma_experiment2.keras
1203/1203 ━━━━━━━━━━━━━━━━━━━━ 455s 211ms/step - accuracy: 0.8044 - auc: 0.8647 - loss: 0.4339 - val_accuracy: 0.8481 - val_auc: 0.9148 - val_loss: 0.3543 - learning_rate: 1.0000e-05
Epoch 2/50
1203/1203 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.8160 - auc: 0.8777 - loss: 0.4171
Epoch 2: val_auc improved from 0.91484 to 0.91809, saving model to /home/jupyter-pushpashrip.23cse/datasets/Glaucoma/Research_Project/projectwork2-phase1/Model_Training/efficientnetb3_glaucoma_experiment2.keras

Epoch 2: finished saving model to /home/jupyter-pushpashrip.23cse/datasets/Glaucoma/Research_Project/projectwork2-

In [26]:
import numpy as np
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# 1. Evaluate on the untouched test set
test_loss, test_accuracy, test_auc = model.evaluate(
    test_dataset,
    verbose=1
)

# 2. Collect true labels and prediction probabilities
y_true = []
y_prob = []

for images, labels in test_dataset:
    probs = model.predict(images, verbose=0).ravel()
    y_prob.extend(probs)
    y_true.extend(labels.numpy())

y_true = np.array(y_true).astype(int)
y_prob = np.array(y_prob)

# 3. Convert probability to class using threshold 0.5
y_pred = (y_prob >= 0.5).astype(int)

# 4. Confusion matrix
cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

# 5. Calculate metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
specificity = tn / (tn + fp)
f1 = f1_score(y_true, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_true, y_prob)

# 6. Display results
print("\n===== EXPERIMENT 2 TEST RESULTS =====")
print(f"Test Loss       : {test_loss:.4f}")
print(f"Accuracy        : {accuracy:.4f}")
print(f"Precision       : {precision:.4f}")
print(f"Recall          : {recall:.4f}")
print(f"Specificity     : {specificity:.4f}")
print(f"F1 Score        : {f1:.4f}")
print(f"ROC-AUC         : {roc_auc:.4f}")

print("\nConfusion Matrix:")
print(cm)

print(f"\nTN: {tn}")
print(f"FP: {fp}")
print(f"FN: {fn}")
print(f"TP: {tp}")

256/258 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.8477 - auc: 0.9224 - loss: 0.3382

E0000 00:00:1788426639.099212 3384345 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1788426649.582373 3384345 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1788426649.838373 3384345 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1788426650.264758 3384345 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1788426650.510827 3384345 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000

258/258 ━━━━━━━━━━━━━━━━━━━━ 74s 287ms/step - accuracy: 0.8468 - auc: 0.9218 - loss: 0.3397

===== EXPERIMENT 2 TEST RESULTS =====
Test Loss       : 0.3397
Accuracy        : 0.8468
Precision       : 0.8358
Recall          : 0.7364
Specificity     : 0.9130
F1 Score        : 0.7830
ROC-AUC         : 0.9219

Confusion Matrix:
[[1176  112]
 [ 204  570]]

TN: 1176
FP: 112
FN: 204
TP: 570
